# Mapa de isolíneas de PPV sobre el plano de mina
### Vibraciones · Parte 2 — Python aplicado a minería · Nilson R. Garrido

En la [primera parte de esta serie](https://nrgarridoa.github.io/articles/vibraciones/) calibramos la **ley de atenuación USBM** `PPV = K · SD^(-β)` y estimamos los parámetros del sitio: **K = 1065 mm/s** y **β = 1.6185** (R² = 0.956 en log-log). Ese modelo es **puntual**: entrega la PPV en función de la distancia escalada, sin ubicarla en el espacio.

El cumplimiento de vibraciones, en cambio, es un problema **espacial y direccional**: un disparo de producción expone simultáneamente a varios receptores (viviendas, instalaciones, labores, taludes), cada uno con un **límite normativo distinto**, a diferentes distancias y azimuts. Este taller lleva la ley calibrada al plano de mina y la convierte en un **campo de PPV**, contorneado en **isolíneas**, con su **frontera de cumplimiento** y las **reglas de carga por retardo** que se derivan. Se trabaja el mapa estático con **Matplotlib** y una versión **interactiva con Plotly**, y se contrasta el modelo homogéneo con un **campo realista** que incorpora heterogeneidad geológica, direccionalidad y topografía.

> **Stack:** Python · NumPy · pandas · Matplotlib · Plotly

## 1) La vibración como problema espacial

La ley calibrada en el Parte 1 predice la PPV para una distancia escalada dada, pero la decisión operativa no es escalar sino **espacial**: antes de disparar, ingeniería de perforación y voladura debe verificar que cada **receptor sensible** quede por debajo de su límite regulatorio. El problema tiene tres rasgos que una evaluación puntual no captura:

- **Es multi-receptor.** Un mismo disparo expone a estructuras distintas (una vivienda, una caseta industrial, el portal de una labor, la cresta de un talud instrumentado), y **cada una responde a un límite normativo diferente** (12.5 mm/s residencial, 50.8 mm/s industrial, criterios geotécnicos), a distancias y direcciones distintas del disparo.
- **Es un campo, no un punto.** La excedencia se localiza: hay una región del plano donde la carga supera el límite y otra donde cumple. Cartografiar esa región es más informativo que una tabla de distancias, y liga la vibración con la **carga operante por retardo** en el diseño del disparo.
- **Es incierto y anisótropo.** La atenuación real no es igual en toda dirección (geología, topografía, orientación de la cara libre). El mapa debe reflejar tanto la **dispersión estadística** del ajuste como la **heterogeneidad del sitio**.

La herramienta que responde a esto es un mapa de **isolíneas de PPV** (curvas de igual velocidad de partícula), análogo a un mapa topográfico pero de vibración. Sobre él se traza la **isolínea del límite** de cada receptor: dentro hay excedencia, fuera hay cumplimiento. Convierte la ley empírica en una **herramienta de planificación y cumplimiento** trazable.

## 2) Marco teórico

### 2.1) El campo espacial de PPV

La ley USBM relaciona la PPV con la **distancia escalada** `SD = D / √W`, donde `D` es la distancia al disparo (m) y `W` la carga máxima por retardo (kg). Sustituyendo en `PPV = K · SD^(-β)`:

$$ PPV(x,y) = K \left(\frac{D(x,y)}{\sqrt{W}}\right)^{-\beta} = K\, W^{\beta/2}\, D(x,y)^{-\beta} $$

Fijada la carga `W`, la PPV en cada punto del plano depende solo de la **distancia a la voladura** `D(x,y)`. Evaluando esa expresión sobre una malla 2D obtenemos un **campo escalar** de PPV que luego contorneamos.

### 2.2) Distancia al polígono del disparo, no a un punto

Un disparo de producción no es un punto: es un **polígono** de taladros (el *round*). Tomar la distancia al centroide sobreestima la PPV cerca del disparo. Usamos la **distancia al borde más cercano del polígono**, de modo que las isolíneas **abrazan la forma real del round** en el campo cercano y se vuelven circulares recién a lo lejos. Dentro del polígono la PPV no está definida (la fórmula diverge cuando `D → 0`), así que enmascaramos su interior.

### 2.3) Incertidumbre: mapa medio y mapa P95

El Parte 1 mostró que los residuos del ajuste son **normales en log₁₀** con desviación σ ≈ 0.110. Eso significa que la PPV real de una voladura se dispersa alrededor de la predicción media. Para diseño **conservador** no se usa la media, sino la **banda superior de predicción**. El percentil 95 unilateral es un factor multiplicativo constante sobre el mapa medio:

$$ PPV_{95}(x,y) = PPV_{media}(x,y)\times 10^{\,1.645\,\sigma} \approx 1.52 \cdot PPV_{media}(x,y) $$

Trabajamos **los dos mapas**: el esperado (media) y el conservador (P95). La diferencia entre ambos es, muchas veces, lo que decide si un receptor cumple o no.

### 2.4) Límites normativos y frontera de cumplimiento

| Norma / Referencia | Límite PPV (mm/s) | Aplicación |
|---|---|---|
| USBM RI 8507 (conservador) | 12.5 | Estructuras residenciales |
| USBM RI 8507 (general) | 50.8 | Estructuras comerciales/industriales |
| NTP 350.004 (Perú) | 50 | Vibraciones por voladura |
| DIN 4150-3 (Alemania) | 5 – 50 | Según estructura y frecuencia |

La **isolínea del límite** de cada receptor separa el plano en zona de cumplimiento y zona de excedencia. Por su forma potencial, esa isolínea es (para fuente puntual) una circunferencia de radio `D_lim = √W · (K / PPV_lim)^{1/β}`.

### 2.5) Reglas de campo en forma cerrada

De la misma ley se despejan dos reglas de uso directo:

- **Distancia mínima segura** para una carga dada:  $D_{min} = \sqrt{W}\,(K/PPV_{lim})^{1/\beta}$
- **Carga máxima por retardo** para un receptor a distancia D:  $W_{max} = (PPV_{lim}/K)^{2/\beta}\,D^{2}$

### 2.6) Limitaciones y el paso al campo realista

El mapa base hereda los supuestos del modelo USBM (carga equivalente por retardo, propagación radial, macizo homogéneo) y añade el de **fuente equivalente en un polígono plano**. Por eso sus isolíneas son **suaves y casi concéntricas**: es fiel al modelo, no una simplificación del dibujo. En un sitio real, en cambio, los contornos son **irregulares y lobulados** por la heterogeneidad geológica (K y β varían en el espacio), la **direccionalidad** de la voladura (la cara libre y la secuencia de iniciación enfocan energía) y la **topografía** (amplificación en crestas). En la Sección 7 integramos estos tres efectos para obtener un campo realista. El mapa base sirve para planificación y cumplimiento; el campo realista requiere calibración con datos de sitio y no sustituye al monitoreo instrumental.

## 3) Datos: el plano de mina

### 3.1) Geometría del escenario

Definimos un sector de rajo sintético en una **grilla local en metros**: el contorno del rajo (crest), el **polígono de disparo** con su carga por retardo, y cuatro **receptores** con sus límites normativos. Todo es determinístico y reproducible.

| Elemento | Valor |
|---|---|
| Extensión del plano | 0 – 1050 m (X) × 0 – 780 m (Y) |
| Carga por retardo `W` | 500 kg (disparo de producción) |
| Ley del sitio (Parte 1) | K = 1065.44 mm/s · β = 1.6185 · σ_log = 0.110 |

### 3.2) Receptores y límites

| Receptor | Tipo | Límite (mm/s) |
|---|---|---|
| Poblado | Residencial (USBM conservador) | 12.5 |
| Caseta de operaciones | Industrial (USBM general) | 50.8 |
| Portal / bocamina | Estructura / labor | 50.0 |
| Cresta de talud | Geotécnico (tolerancia alta) | 100.0 |

## 4) Implementación en Python

### 4.1) Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path
import plotly.graph_objects as go

# Ley del sitio heredada del Parte 1 (ajuste USBM)
K_SITE   = 1065.44     # mm/s
BETA     = 1.6185      # exponente de atenuacion
SIGMA_LOG = 0.110      # desviacion de residuos en log10 (Parte 1)
F_P95    = 10 ** (1.645 * SIGMA_LOG)   # factor medio -> P95 (~1.52)
W_DELAY  = 500.0       # carga maxima por retardo (kg)

print(f'Factor P95 (medio -> conservador): x{F_P95:.3f}')

### 4.2) Geometría del plano y receptores

El polígono de disparo se define por sus vértices; el contorno del rajo se genera de forma paramétrica. Guardamos el polígono y los receptores en `data/raw/` para el repositorio.

In [ ]:
# Poligono de disparo (round), coords locales en m
blast = np.array([(240, 310), (370, 300), (380, 400), (255, 405)], dtype=float)

# Contorno del rajo (crest) parametrico, solo contexto visual
th = np.linspace(0, 2 * np.pi, 200)
pit_x = 430 + 360 * np.cos(th) + 40 * np.cos(3 * th)
pit_y = 380 + 300 * np.sin(th) + 30 * np.sin(2 * th)
pit = np.column_stack([pit_x, pit_y])

# Receptores: nombre, coords, limite normativo (mm/s)
receptores = pd.DataFrame([
    ('Poblado',               720, 610, 12.5, 'Residencial'),
    ('Caseta de operaciones',  60, 120, 50.8, 'Industrial'),
    ('Portal / bocamina',     640, 150, 50.0, 'Estructura'),
    ('Cresta de talud',       230, 530, 100.0, 'Geotecnico'),
], columns=['receptor', 'x', 'y', 'limite_mm_s', 'tipo'])

import os
os.makedirs('../data/raw', exist_ok=True)
pd.DataFrame(blast, columns=['x', 'y']).to_csv('../data/raw/blast_polygon.csv', index=False)
receptores.to_csv('../data/raw/receptores.csv', index=False)
receptores

### 4.3) Distancia al polígono del disparo (vectorizada)

Para cada punto calculamos la distancia mínima a los **segmentos** del polígono. La función opera sobre toda la malla a la vez.

In [ ]:
def dist_a_poligono(px, py, poly):
    """Distancia de cada punto (px, py) al borde del poligono (min sobre segmentos)."""
    P = np.column_stack([px.ravel(), py.ravel()])
    best = np.full(P.shape[0], np.inf)
    n = len(poly)
    for i in range(n):
        a, b = poly[i], poly[(i + 1) % n]
        ab = b - a
        t = np.clip(((P - a) @ ab) / (ab @ ab), 0.0, 1.0)
        proj = a + t[:, None] * ab
        d = np.linalg.norm(P - proj, axis=1)
        best = np.minimum(best, d)
    return best.reshape(px.shape)

# Distancia de un receptor puntual (para las tablas)
def dist_receptor(x, y):
    return float(dist_a_poligono(np.array([[x]], float), np.array([[y]], float), blast)[0, 0])

### 4.4) Campo de PPV sobre la malla

Evaluamos `PPV(x,y) = K · W^(β/2) · D^(-β)` en cada nodo. Enmascaramos el interior del polígono (la fórmula diverge cuando `D → 0`) y calculamos también el campo conservador P95.

In [ ]:
NX, NY = 420, 320
xx = np.linspace(0, 1050, NX)
yy = np.linspace(0, 780, NY)
XX, YY = np.meshgrid(xx, yy)

D = dist_a_poligono(XX, YY, blast)

# Enmascarar el interior del poligono y poner un piso fisico de 8 m
# (la formula diverge cuando D -> 0; nadie mide a 0 m del disparo)
dentro = Path(blast).contains_points(np.column_stack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
D = np.where(dentro, np.nan, np.maximum(D, 8.0))

def campo_ppv(D, W=W_DELAY):
    return K_SITE * (W ** (BETA / 2)) * D ** (-BETA)

PPV_media = campo_ppv(D)
PPV_p95   = PPV_media * F_P95

print(f'PPV media  -> min {np.nanmin(PPV_media):.2f} | max {np.nanmax(PPV_media):.1f} mm/s')
print(f'PPV P95    -> min {np.nanmin(PPV_p95):.2f} | max {np.nanmax(PPV_p95):.1f} mm/s')

## 5) Mapa de isolíneas con Matplotlib

### 5.1) Mapa medio con frontera de cumplimiento

Rellenamos el campo con una escala de color, trazamos isolíneas etiquetadas en mm/s y resaltamos la **isolínea de 12.5 mm/s** (límite residencial) como frontera de cumplimiento. Superponemos el polígono de disparo y los receptores.

In [ ]:
NIVELES = [2, 5, 12.5, 25, 50.8, 100, 200]              # umbrales normativos (relleno + etiqueta)
NIVELES_FINOS = [2, 3.5, 5, 8, 12.5, 18, 25, 35, 50.8, 70, 100, 140, 200]   # curvas intermedias (lectura)

def dibujar_mapa(ax, campo, titulo):
    cf = ax.contourf(XX, YY, campo, levels=NIVELES, cmap='YlOrRd', extend='max', alpha=0.85)
    ax.contour(XX, YY, campo, levels=NIVELES_FINOS, colors='#5f4a3a', linewidths=0.45, alpha=0.5)
    cs = ax.contour(XX, YY, campo, levels=[5, 12.5, 25, 50.8, 100], colors='#5f4a3a', linewidths=0)
    ax.clabel(cs, fmt='%g', fontsize=8)
    # frontera de cumplimiento residencial (12.5 mm/s)
    ax.contour(XX, YY, campo, levels=[12.5], colors='#111827', linewidths=2.4)
    # rajo y disparo
    ax.plot(pit[:, 0], pit[:, 1], color='#6b7280', lw=1.2, ls='--', alpha=0.8)
    ax.fill(blast[:, 0], blast[:, 1], facecolor='#111827', edgecolor='white', lw=1.2, zorder=5)
    ax.text(blast[:, 0].mean(), blast[:, 1].mean(), 'DISPARO', color='white',
            ha='center', va='center', fontsize=7.5, fontweight='bold', zorder=6)
    # receptores
    for _, r in receptores.iterrows():
        ax.scatter(r.x, r.y, s=90, marker='^', c='#0f766e', edgecolors='white', linewidths=1.2, zorder=7)
        ax.annotate(f"{r.receptor}\n(lim {r.limite_mm_s:g})", (r.x, r.y),
                    textcoords='offset points', xytext=(9, 6), fontsize=8, fontweight='bold',
                    color='#0f2c25', zorder=8)
    ax.set_xlabel('Este local (m)'); ax.set_ylabel('Norte local (m)')
    ax.set_title(titulo); ax.set_aspect('equal'); ax.set_xlim(0, 1050); ax.set_ylim(0, 780)
    return cf

fig, ax = plt.subplots(figsize=(12, 8.5))
cf = dibujar_mapa(ax, PPV_media, f'Mapa de PPV media  ·  W = {W_DELAY:.0f} kg/retardo  ·  frontera 12.5 mm/s (linea negra)')
cb = fig.colorbar(cf, ax=ax, shrink=0.85, label='PPV (mm/s)')
plt.tight_layout(); plt.show()

La lectura es inmediata: la **línea negra** (12.5 mm/s) encierra la zona donde una estructura residencial excedería el límite. Todos los receptores caen fuera de ella en el mapa medio. Pero la media no es el criterio de diseño.

### 5.2) Mapa medio vs P95: la frontera se expande

El mapa conservador (P95) desplaza la frontera de cumplimiento hacia afuera. Comparamos la isolínea de 12.5 mm/s en ambos escenarios.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8.5))
ax.contourf(XX, YY, PPV_p95, levels=NIVELES, cmap='YlOrRd', extend='max', alpha=0.5)
ax.contour(XX, YY, PPV_p95, levels=NIVELES_FINOS, colors='#5f4a3a', linewidths=0.4, alpha=0.35)
c_med = ax.contour(XX, YY, PPV_media, levels=[12.5], colors='#111827', linewidths=2.4)
c_p95 = ax.contour(XX, YY, PPV_p95,   levels=[12.5], colors='#b4312a', linewidths=2.4, linestyles='--')
ax.plot(pit[:, 0], pit[:, 1], color='#6b7280', lw=1.2, ls='--', alpha=0.8)
ax.fill(blast[:, 0], blast[:, 1], facecolor='#111827', edgecolor='white', lw=1.2, zorder=5)
for _, r in receptores.iterrows():
    ax.scatter(r.x, r.y, s=90, marker='^', c='#0f766e', edgecolors='white', linewidths=1.2, zorder=7)
    ax.annotate(r.receptor, (r.x, r.y), textcoords='offset points', xytext=(9, 6),
                fontsize=8, fontweight='bold', color='#0f2c25', zorder=8)
h = [plt.Line2D([], [], color='#111827', lw=2.4, label='Frontera 12.5 mm/s (media)'),
     plt.Line2D([], [], color='#b4312a', lw=2.4, ls='--', label='Frontera 12.5 mm/s (P95)')]
ax.legend(handles=h, loc='upper left')
ax.set_xlabel('Este local (m)'); ax.set_ylabel('Norte local (m)')
ax.set_title('Frontera de cumplimiento residencial: media vs P95'); ax.set_aspect('equal')
ax.set_xlim(0, 1050); ax.set_ylim(0, 780)
plt.tight_layout(); plt.show()

El **poblado** queda fuera de la frontera media pero **dentro de la frontera P95**: en el escenario esperado cumple, en el conservador excede. Ese es el punto que decide el diseño.

## 6) Evaluación de receptores

Calculamos la distancia de cada receptor al disparo y su PPV media y P95, contra el límite propio.

In [ ]:
filas = []
for _, r in receptores.iterrows():
    Dr = dist_receptor(r.x, r.y)
    pm = campo_ppv(np.array(Dr)).item()
    pp = pm * F_P95
    filas.append({'receptor': r.receptor, 'D_m': round(Dr), 'PPV_media': round(pm, 1),
                  'PPV_P95': round(pp, 1), 'limite': r.limite_mm_s,
                  'cumple_media': 'OK' if pm < r.limite_mm_s else 'EXCEDE',
                  'cumple_P95':   'OK' if pp < r.limite_mm_s else 'EXCEDE'})
eval_df = pd.DataFrame(filas)
eval_df

El receptor de **mayor PPV** es la cresta de talud (127 m del disparo), pero su tolerancia geotécnica (100 mm/s) la mantiene holgada. El **vinculante** es el poblado: la PPV más baja de la tabla, pero con el límite más estricto y en el escenario P95 lo supera. La restricción de diseño **no la fija el punto más cercano ni el de mayor vibración, sino el de límite más exigente frente a su exposición**.

## 7) El campo realista: heterogeneidad, direccionalidad y topografía

El mapa base supone un macizo homogéneo y una fuente radial, y por eso sus isolíneas salen suaves. Un campo real es **anisótropo**. Modelamos esa realidad con un **factor de sitio** `M(x,y)` que multiplica la PPV media e integra tres efectos, cada uno documentado y reproducible:

$$ PPV_{real}(x,y) = PPV_{media}(x,y)\cdot M(x,y), \qquad M = M_{dir}\cdot M_{geo}\cdot M_{topo} $$

- **Direccionalidad** `M_dir`: la cara libre y la secuencia de iniciación enfocan energía en una dirección. Se modela como `1 + A·cos(θ − θ₀)`, con `θ₀` el azimut de la cara libre (aquí, hacia el poblado) y `A = 0.22` (±22 % entre el frente y la espalda del disparo).
- **Heterogeneidad geológica** `M_geo`: K y β varían en el espacio (litologías, fracturamiento, agua). Se representa con un campo suave de baja frecuencia (semilla fija) más una **zona más fracturada** que amplifica hacia el poblado.
- **Topografía** `M_topo`: amplificación en una banda a lo largo de la cresta del rajo (+15 %).

Los parámetros son ilustrativos: en un caso real, `M(x,y)` se calibra con registros de geófonos distribuidos y, típicamente, interpolando (kriging) las mediciones de varias voladuras.

In [ ]:
cen = blast.mean(axis=0)                        # centroide del disparo
th0 = np.arctan2(610 - cen[1], 720 - cen[0])    # azimut disparo -> poblado (cara libre)
A_DIR = 0.22                                     # amplitud direccional (+-22%)

def modificador(X, Y):
    # 1) direccionalidad (cara libre hacia el poblado)
    m_dir = 1 + A_DIR * np.cos(np.arctan2(Y - cen[1], X - cen[0]) - th0)
    # 2) heterogeneidad geologica: campo suave + zona fracturada hacia el poblado
    m_geo = np.ones_like(X, dtype=float)
    rng = np.random.default_rng(11)
    for _ in range(5):
        ax_, ay_ = rng.uniform(0.4, 1.4, 2) / 1000; ph = rng.uniform(0, 2 * np.pi, 2)
        m_geo += rng.uniform(0.04, 0.08) * np.sin(ax_ * X + ph[0]) * np.cos(ay_ * Y + ph[1])
    m_geo *= 1 + 0.15 * np.exp(-(((X - 720) / 380) ** 2 + ((Y - 600) / 320) ** 2))
    # 3) topografia: amplificacion en la banda de la cresta
    pd_ = np.min(np.sqrt((X[..., None] - pit[:, 0]) ** 2 + (Y[..., None] - pit[:, 1]) ** 2), axis=-1)
    m_topo = 1 + 0.15 * np.exp(-(pd_ / 50) ** 2)
    return m_dir * m_geo * m_topo

M = modificador(XX, YY)
PPV_real = PPV_media * M
print(f'Factor de sitio M: min {np.nanmin(M):.2f} | max {np.nanmax(M):.2f}')

### 7.1) Isolíneas del campo homogéneo vs el realista

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), sharey=True)
for ax, campo, tit in [(axes[0], PPV_media, 'Modelo homogéneo (ley USBM)'),
                       (axes[1], PPV_real, 'Campo realista (geología + dirección + topografía)')]:
    ax.contourf(XX, YY, campo, levels=NIVELES, cmap='YlOrRd', extend='max', alpha=0.85)
    ax.contour(XX, YY, campo, levels=NIVELES_FINOS, colors='#5f4a3a', linewidths=0.45, alpha=0.5)
    ax.contour(XX, YY, campo, levels=[12.5], colors='#111827', linewidths=2.4)
    ax.plot(pit[:, 0], pit[:, 1], color='#6b7280', lw=1.1, ls='--', alpha=0.8)
    ax.fill(blast[:, 0], blast[:, 1], facecolor='#111827', edgecolor='white', lw=1.1, zorder=5)
    for _, r in receptores.iterrows():
        ax.scatter(r.x, r.y, s=80, marker='^', c='#0f766e', edgecolors='white', linewidths=1.1, zorder=7)
        ax.annotate(r.receptor, (r.x, r.y), textcoords='offset points', xytext=(7, 5), fontsize=8,
                    fontweight='bold', color='#0f2c25', zorder=8)
    ax.set_xlabel('Este local (m)'); ax.set_aspect('equal'); ax.set_xlim(0, 1050); ax.set_ylim(0, 780)
    ax.set_title(tit, fontsize=11)
axes[0].set_ylabel('Norte local (m)')
plt.tight_layout(); plt.show()

Las isolíneas del campo realista son **lobuladas**: la frontera de 12.5 mm/s se **abulta hacia el poblado** (dirección de la cara libre y zona más fracturada) y se **contrae hacia la caseta** (a la espalda del disparo). El punto de mayor PPV ya no coincide con el disparo, y la forma deja de ser radial.

### 7.2) Receptores bajo el campo realista

In [ ]:
filas = []
for _, r in receptores.iterrows():
    Dr = dist_receptor(r.x, r.y)
    pm = campo_ppv(np.array(Dr)).item()
    mr = modificador(np.array([[float(r.x)]]), np.array([[float(r.y)]]))[0, 0]
    pr = pm * mr
    filas.append({'receptor': r.receptor, 'PPV_homogenea': round(pm, 1), 'M': round(mr, 2),
                  'PPV_realista': round(pr, 1), 'limite': r.limite_mm_s,
                  'cumple': 'OK' if pr < r.limite_mm_s else 'EXCEDE'})
pd.DataFrame(filas)

El resultado refuerza la conclusión y la endurece: con la **direccionalidad hacia el poblado** más la **zona fracturada**, su PPV realista sube de 10.0 a **16.7 mm/s** y **excede el límite ya en la media** (sin necesidad del P95). La caseta, a la espalda del disparo, se **atenúa**. Es decir, el mapa homogéneo **subestima** el riesgo en la dirección crítica: cuando el sitio es anisótropo, el diseño conservador no es opcional, y el receptor a favor de la cara libre manda todavía más.

## 8) Mapa interactivo con Plotly

La versión interactiva permite pasar el cursor por cualquier punto y leer la PPV, hacer zoom y aislar receptores. Es la que se embebe en el artículo web.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Contour(
    x=xx, y=yy, z=PPV_media,
    colorscale='YlOrRd', zmin=0, zmax=100,
    contours=dict(start=0, end=100, size=12.5, showlabels=True,
                  labelfont=dict(size=10, color='#3b2a1a')),
    colorbar=dict(title='PPV<br>(mm/s)'),
    hovertemplate='Este %{x:.0f} m<br>Norte %{y:.0f} m<br>PPV %{z:.1f} mm/s<extra></extra>'))

# frontera de cumplimiento 12.5 mm/s
fig.add_trace(go.Contour(
    x=xx, y=yy, z=PPV_media, showscale=False, contours=dict(start=12.5, end=12.5, size=1, coloring='none'),
    line=dict(color='#111827', width=3), hoverinfo='skip', name='12.5 mm/s'))

# poligono de disparo
fig.add_trace(go.Scatter(x=np.append(blast[:, 0], blast[0, 0]), y=np.append(blast[:, 1], blast[0, 1]),
    fill='toself', fillcolor='#111827', line=dict(color='white'), mode='lines',
    name='Disparo', hoverinfo='name'))

# receptores
fig.add_trace(go.Scatter(
    x=receptores.x, y=receptores.y, mode='markers+text',
    marker=dict(size=13, color='#0f766e', symbol='triangle-up', line=dict(color='white', width=1.5)),
    text=receptores.receptor, textposition='top center', textfont=dict(size=11),
    customdata=receptores.limite_mm_s,
    hovertemplate='%{text}<br>Limite %{customdata:.1f} mm/s<extra></extra>', name='Receptores'))

fig.update_layout(
    title=f'PPV sobre el plano de mina  ·  W = {W_DELAY:.0f} kg/retardo',
    xaxis_title='Este local (m)', yaxis_title='Norte local (m)',
    width=920, height=680, template='plotly_white')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

Para embeber el mapa en una página web se exporta como fragmento HTML autocontenido:

```python
fig.write_html('../ppv_interactivo.html', include_plotlyjs='cdn', full_html=True)
```

## 9) Reglas de campo

### 9.1) Distancia mínima segura

Dado un límite y una carga por retardo, `D_min = √W · (K / PPV_lim)^(1/β)`. Es el radio de la isolínea del límite (mapa medio).

In [ ]:
limites = [50.8, 25, 12.5, 5]
cargas  = [100, 250, 500, 750, 1000]

print('Distancia minima segura D_min (m) -- mapa medio')
print('=' * 62)
print(f"{'W (kg)':>8}" + ''.join(f'  PPV<{L:>5} mm/s' for L in limites))
print('-' * 62)
for W in cargas:
    fila = f'{W:>8}'
    for L in limites:
        d_min = np.sqrt(W) * (K_SITE / L) ** (1 / BETA)
        fila += f'{d_min:>16.0f}'
    print(fila)

### 9.2) Carga máxima por retardo por receptor

`W_max = (PPV_lim / K)^(2/β) · D²`. Es la regla que usa el supervisor: dada la distancia al receptor y su límite, la carga máxima que puede detonar por retardo. La calculamos en el escenario **medio** y en el **conservador P95**.

In [ ]:
filas = []
for _, r in receptores.iterrows():
    Dr = dist_receptor(r.x, r.y)
    w_med = (r.limite_mm_s / K_SITE) ** (2 / BETA) * Dr ** 2
    w_p95 = (r.limite_mm_s / (K_SITE * F_P95)) ** (2 / BETA) * Dr ** 2
    filas.append({'receptor': r.receptor, 'D_m': round(Dr), 'limite': r.limite_mm_s,
                  'W_max_media_kg': round(w_med), 'W_max_P95_kg': round(w_p95)})
wmax_df = pd.DataFrame(filas)
print(wmax_df.to_string(index=False))
w_binding = wmax_df['W_max_P95_kg'].min()
quien = wmax_df.loc[wmax_df['W_max_P95_kg'].idxmin(), 'receptor']
print(f'\nCarga maxima por retardo que satisface a TODOS los receptores (P95): {w_binding:.0f} kg')
print(f'Receptor vinculante: {quien}')

### 9.3) Rediseño: disparo que cumple al P95

El disparo actual (500 kg/retardo) satisface a todos en la media, pero el poblado excede al P95. La carga máxima que cumple con **todos los receptores en el escenario conservador** es la del receptor vinculante. Aplicamos una carga de diseño con margen y volvemos a mapear.

In [ ]:
W_DISENO = 350.0   # kg/retardo, con margen sobre el limite P95 del poblado (391 kg)

PPV_rediseno = campo_ppv(D, W=W_DISENO) * F_P95   # mapa P95 con la carga de diseno

fig, ax = plt.subplots(figsize=(12, 8.5))
cf = ax.contourf(XX, YY, PPV_rediseno, levels=NIVELES, cmap='YlGn', extend='max', alpha=0.85)
ax.contour(XX, YY, PPV_rediseno, levels=[12.5], colors='#111827', linewidths=2.4)
ax.plot(pit[:, 0], pit[:, 1], color='#6b7280', lw=1.2, ls='--', alpha=0.8)
ax.fill(blast[:, 0], blast[:, 1], facecolor='#111827', edgecolor='white', lw=1.2, zorder=5)
for _, r in receptores.iterrows():
    Dr = dist_receptor(r.x, r.y); pp = campo_ppv(np.array(Dr), W=W_DISENO).item() * F_P95
    ok = pp < r.limite_mm_s
    ax.scatter(r.x, r.y, s=95, marker='^', c=('#0f766e' if ok else '#b4312a'),
               edgecolors='white', linewidths=1.2, zorder=7)
    ax.annotate(f'{r.receptor}\n{pp:.1f} mm/s', (r.x, r.y), textcoords='offset points',
                xytext=(9, 6), fontsize=8, fontweight='bold', color='#0f2c25', zorder=8)
cb = fig.colorbar(cf, ax=ax, shrink=0.85, label='PPV P95 (mm/s)')
ax.set_xlabel('Este local (m)'); ax.set_ylabel('Norte local (m)')
ax.set_title(f'Disparo rediseñado: W = {W_DISENO:.0f} kg/retardo (mapa P95, todos cumplen)')
ax.set_aspect('equal'); ax.set_xlim(0, 1050); ax.set_ylim(0, 780)
plt.tight_layout(); plt.show()

Con 350 kg/retardo la frontera de cumplimiento se contrae lo suficiente para que **el poblado quede fuera incluso en el mapa P95**. El costo es operativo (más retardos, disparos más pequeños o más secuenciados), y esa es la compensación que el mapa pone sobre la mesa con evidencia.

## 10) Conclusiones operativas

- La ley USBM ajustada en el Parte 1 (**K = 1065, β = 1.6185**) se lleva al plano como un **campo de PPV** y se contornea en isolíneas: el mapa muestra de un vistazo dónde una carga cumple y dónde excede.
- Usar la **distancia al polígono del disparo** (no al centroide) hace que las isolíneas cercanas sigan la forma real del round; a lo lejos tienden a circunferencias.
- El **mapa P95** (× 1.52 sobre la media, derivado de σ = 0.110 del Parte 1) desplaza la frontera de cumplimiento hacia afuera. Diseñar con la media subestima el riesgo; el criterio conservador usa el P95.
- La restricción de diseño la fija el **receptor vinculante**: en este caso el poblado, que tiene la PPV más baja pero el límite más estricto. No es ni el más cercano ni el de mayor vibración.
- Bajar la carga de **500 a 350 kg/retardo** devuelve el cumplimiento del poblado en el escenario P95. El mapa convierte la ley de atenuación en una **decisión de carga por retardo** trazable y auditable.
- El **campo realista** (heterogeneidad + direccionalidad + topografía) deforma las isolíneas y, en la dirección de la cara libre, lleva al poblado a exceder ya en la media (16.7 mm/s): el modelo homogéneo subestima el riesgo donde la geometría del disparo enfoca la energía.

El flujo es reproducible de principio a fin y se conecta con el monitoreo real: cada campaña recalibra K y β (Parte 1), el factor de sitio M(x,y) (Sección 7) y actualiza el mapa (Parte 2).

## 11) Referencias

- Siskind, D. E., Stagg, M. S., Kopp, J. W., & Dowding, C. H. (1980). *Structure response and damage produced by ground vibration from surface mine blasting.* U.S. Bureau of Mines, RI 8507.
- Dowding, C. H. (1985). *Blast Vibration Monitoring and Control.* Prentice-Hall.
- Agrawal, H., & Mishra, A. K. (2019). Modified scaled distance regression analysis approach for prediction of blast-induced ground vibration in multi-hole blasting. *JRMGE*, 11, 202–207.
- Hustrulid, W. (1999). *Blasting Principles for Open Pit Mining.* Balkema.
- DIN 4150-3 (2016). *Structural vibration — Effects of vibration on structures.*

---
*Nilson Rolando Garrido Asenjo · Ingeniero de Minas · Python para Minería · [linkedin.com/in/nrgarridoa](https://www.linkedin.com/in/nrgarridoa) · [nrgarridoa.github.io](https://nrgarridoa.github.io)*